# 🚀 IDFC Invoice Data Extraction - Supervised & VLM/CNN Models

This notebook provides a clean, modern pipeline for invoice field extraction using either a CNN (ResNet50) or a Vision-Language Model (CLIP).
- **No pseudo-labeling**: Only labeled data is used for training and testing.
- **Model selection**: Easily switch between CNN and CLIP+MLP for best results.
- **Per-field and per-image accuracy**: Full transparency and detailed evaluation.

---

In [ ]:
# Check GPU and environment (optional, can comment out if not needed)
# List all processes using the GPU
!fuser -v /dev/nvidia*
!nvidia-smi || echo "No GPU available" --- IGNORE ---



Thu Jan 22 21:18:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ============================================================================
# CELL 1: INSTALL DEPENDENCIES FOR TRAINING
# ============================================================================
!pip install -q torch torchvision transformers datasets accelerate
!pip install -q easyocr rapidfuzz opencv-python-headless pillow
!pip install -q scikit-learn matplotlib tqdm

import torch
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🎮 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📱 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    # Clear GPU memory
    torch.cuda.empty_cache()
    print(f"🧹 Free VRAM: {torch.cuda.memory_reserved(0) / 1024**3:.1f} GB reserved")

🔥 PyTorch: 2.9.0+cu126
🎮 CUDA Available: True
📱 GPU: Tesla T4
💾 VRAM: 14.7 GB
🧹 Free VRAM: 0.0 GB reserved


In [ ]:
# ============================================================================
# CELL 2: IMPORTS & CONFIGURATION
# ============================================================================

import os
import sys
import json
import time
import random
import re
import warnings
import logging
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from collections import defaultdict
from tqdm import tqdm

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from rapidfuzz import fuzz, process

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🐍 Python: {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"📱 Device: {DEVICE}")

# ============================================================================
# PATH CONFIGURATION  
# ============================================================================
POSSIBLE_PATHS = [
    Path('/content/GenAI--Invoice-data-extractor-cnn'),  # Colab
    Path('/workspaces/GenAI--Invoice-data-extractor-cnn'),  # Codespaces
    Path.cwd().parent,
]

BASE_DIR = None
for p in POSSIBLE_PATHS:
    if (p / 'train').exists():
        BASE_DIR = p
        break

if BASE_DIR is None:
    raise FileNotFoundError("Repository not found!")

TRAIN_DIR = BASE_DIR / 'train'
GT_PATH = BASE_DIR / 'data-ground_truth' / 'output' / 'result.json'
OUTPUT_DIR = BASE_DIR / 'output'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Base: {BASE_DIR}")
print(f"📸 Images: {len(list(TRAIN_DIR.glob('*.png')))}")

🐍 Python: 3.12.12
🔥 PyTorch: 2.9.0+cu126
📱 Device: cuda

📂 Base: /content/GenAI--Invoice-data-extractor-cnn
📸 Images: 495


In [ ]:
# ============================================================================
# CELL 3: LOAD DATA WITH TRAIN/TEST SPLIT (SUPERVISED ONLY, NO UNLABELED DATA)
# ============================================================================

def load_data_with_split(gt_path: Path, train_dir: Path, test_ratio: float = 0.35, seed: int = 42):
    """Load and split data into train/test sets. Only use labeled data."""
    with open(gt_path, 'r', encoding='utf-8') as f:
        gt_list = json.load(f)
    
    # Parse ground truth
    all_gt = {}
    for item in gt_list:
        for filename, data in item.items():
            all_gt[Path(filename).name] = data['fields']
    
    # Get all images
    all_images = {p.name: p for p in train_dir.glob('*.png')}
    all_images.update({p.name: p for p in train_dir.glob('*.jpg')})
    
    # Only use images with ground truth
    labeled_files = [f for f in all_gt.keys() if f in all_images]
    random.seed(seed)
    random.shuffle(labeled_files)
    
    split_idx = int(len(labeled_files) * (1 - test_ratio))
    train_files = labeled_files[:split_idx]
    test_files = labeled_files[split_idx:]
    
    # Create datasets
    train_data = [(all_images[f], all_gt[f]) for f in train_files]
    test_data = [(all_images[f], all_gt[f]) for f in test_files]
    
    # Extract unique values for classification
    dealers = list(set(gt['dealer_name'] for _, gt in train_data if gt.get('dealer_name')))
    models_list = list(set(gt['model_name'] for _, gt in train_data if gt.get('model_name')))
    
    return train_data, test_data, dealers, models_list

# Load data (supervised only)
train_data, test_data, KNOWN_DEALERS, KNOWN_MODELS = load_data_with_split(GT_PATH, TRAIN_DIR)

print(f"\n{'='*60}")
print(f"📊 DATA LOADED FOR SUPERVISED LEARNING")
print(f"{'='*60}")
print(f"🏷️  Labeled Training:   {len(train_data)} images")
print(f"🧪 Labeled Testing:    {len(test_data)} images")  
print(f"{'─'*60}")
print(f"Unique dealers: {len(KNOWN_DEALERS)}")
print(f"Unique models: {len(KNOWN_MODELS)}")


📊 DATA LOADED FOR SUPERVISED LEARNING
🏷️  Labeled Training:   141 images
🧪 Labeled Testing:    77 images
────────────────────────────────────────────────────────────
Unique dealers: 119
Unique models: 134


---

**Note on Accuracy and Data Split:**

- The code strictly splits labeled data into training and test sets using a random seed for reproducibility.
- Only the training set is used for model fine-tuning. The test set is held out and never used for training.
- Final accuracy is computed on the test set by comparing predictions to ground truth for each field, ensuring a fair evaluation.
- Per-field and per-image accuracy are reported for transparency.

---

In [ ]:
# ============================================================================
# CELL 4: INVOICE DATASET CLASS
# ============================================================================

class InvoiceDataset(Dataset):
    """Dataset for invoice field extraction with data augmentation."""
    
    def __init__(self, data: List[Tuple[Path, Dict]], dealer_encoder: LabelEncoder, 
                 model_encoder: LabelEncoder, transform=None, is_train=True):
        self.data = data
        self.dealer_encoder = dealer_encoder
        self.model_encoder = model_encoder
        self.is_train = is_train
        
        # Transforms
        if transform is None:
            if is_train:
                self.transform = transforms.Compose([
                    transforms.Resize((384, 384)),
                    transforms.RandomRotation(5),
                    transforms.ColorJitter(brightness=0.2, contrast=0.2),
                    transforms.ToTensor(),
                    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])
            else:
                self.transform = transforms.Compose([
                    transforms.Resize((384, 384)),
                    transforms.ToTensor(),
                    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])
        else:
            self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, fields = self.data[idx]
        
        # Load image
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)
        
        # Encode labels
        dealer_name = fields.get('dealer_name', 'Unknown')
        model_name = fields.get('model_name', 'Unknown')
        
        # Handle unknown labels
        try:
            dealer_idx = self.dealer_encoder.transform([dealer_name])[0]
        except:
            dealer_idx = 0  # Unknown class
        
        try:
            model_idx = self.model_encoder.transform([model_name])[0]
        except:
            model_idx = 0
        
        # Numeric fields
        hp = fields.get('horse_power') or 0
        hp = float(hp) if hp else 0
        
        cost = fields.get('asset_cost') or 0
        cost = float(cost) if cost else 0
        
        # Binary fields
        sig = 1.0 if fields.get('signature', {}).get('present', False) else 0.0
        stamp = 1.0 if fields.get('stamp', {}).get('present', False) else 0.0
        
        return {
            'image': img,
            'dealer_idx': torch.tensor(dealer_idx, dtype=torch.long),
            'model_idx': torch.tensor(model_idx, dtype=torch.long),
            'horse_power': torch.tensor(hp / 100.0, dtype=torch.float),  # Normalized
            'asset_cost': torch.tensor(cost / 1000000.0, dtype=torch.float),  # Normalized
            'signature': torch.tensor(sig, dtype=torch.float),
            'stamp': torch.tensor(stamp, dtype=torch.float),
            'filename': img_path.name
        }

# Create label encoders
dealer_encoder = LabelEncoder()
model_encoder = LabelEncoder()

# Fit on training data (add 'Unknown' for unseen labels)
all_dealers = ['Unknown'] + KNOWN_DEALERS
all_models = ['Unknown'] + KNOWN_MODELS
dealer_encoder.fit(all_dealers)
model_encoder.fit(all_models)

print(f"✅ Label encoders created")
print(f"   Dealer classes: {len(dealer_encoder.classes_)}")
print(f"   Model classes: {len(model_encoder.classes_)}")

✅ Label encoders created
   Dealer classes: 120
   Model classes: 135


In [ ]:
# ============================================================================
# CELL: CLEAR GPU MEMORY BEFORE MODEL CREATION
# ============================================================================
import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("🧹 Cleared GPU memory before model creation.")

🧹 Cleared GPU memory before model creation.


In [ ]:
# =============================
# Suggestions to Further Improve Accuracy
# =============================

def suggest_accuracy_improvements():
    print("\nSuggestions to further improve accuracy:")
    print("- Try more advanced data augmentations (CutMix, MixUp, random erasing, etc.)")
    print("- Use test-time augmentation (TTA) for more robust predictions")
    print("- Ensemble multiple models (e.g., CNN + CLIP)")
    print("- Tune hyperparameters (learning rate, batch size, optimizer)")
    print("- Use more advanced architectures (e.g., Swin Transformer, ViT, BLIP)")
    print("- Apply label smoothing or focal loss for imbalanced fields")
    print("- Use pseudo-labeling or semi-supervised learning if more unlabeled data is available")
    print("- Perform error analysis to identify and address common failure cases")

# Call this function to print suggestions
suggest_accuracy_improvements()


Suggestions to further improve accuracy:
- Try more advanced data augmentations (CutMix, MixUp, random erasing, etc.)
- Use test-time augmentation (TTA) for more robust predictions
- Ensemble multiple models (e.g., CNN + CLIP)
- Tune hyperparameters (learning rate, batch size, optimizer)
- Use more advanced architectures (e.g., Swin Transformer, ViT, BLIP)
- Apply label smoothing or focal loss for imbalanced fields
- Use pseudo-labeling or semi-supervised learning if more unlabeled data is available
- Perform error analysis to identify and address common failure cases


In [ ]:
# ============================================================================
# CELL 5: MULTI-TASK CNN MODEL (ResNet50 backbone)
# ============================================================================

class InvoiceExtractorCNN(nn.Module):
    """
    Multi-task CNN for invoice field extraction.
    Uses ResNet50 backbone with multiple output heads.
    """
    
    def __init__(self, num_dealers: int, num_models: int, dropout: float = 0.3):
        super().__init__()
        
        # Backbone: ResNet50 pretrained on ImageNet
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        
        # Feature dimension from ResNet50
        feat_dim = 2048
        
        # Shared layers
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
        # Task-specific heads
        self.dealer_head = nn.Linear(512, num_dealers)      # Classification
        self.model_head = nn.Linear(512, num_models)        # Classification
        self.hp_head = nn.Linear(512, 1)                    # Regression
        self.cost_head = nn.Linear(512, 1)                  # Regression
        self.signature_head = nn.Linear(512, 1)             # Binary
        self.stamp_head = nn.Linear(512, 1)                 # Binary
        
    def forward(self, x):
        # Extract features
        features = self.features(x)
        shared = self.shared(features)
        
        return {
            'dealer': self.dealer_head(shared),
            'model': self.model_head(shared),
            'horse_power': self.hp_head(shared).squeeze(-1),
            'asset_cost': self.cost_head(shared).squeeze(-1),
            'signature': torch.sigmoid(self.signature_head(shared)).squeeze(-1),
            'stamp': torch.sigmoid(self.stamp_head(shared)).squeeze(-1),
        }

# Create model
num_dealers = len(dealer_encoder.classes_)
num_models = len(model_encoder.classes_)

model = InvoiceExtractorCNN(num_dealers, num_models).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🧠 Model created: InvoiceExtractorCNN")
print(f"   Backbone: ResNet50 (pretrained)")
print(f"   Dealer classes: {num_dealers}")
print(f"   Model classes: {num_models}")
print(f"   Total params: {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")

🧠 Model created: InvoiceExtractorCNN
   Backbone: ResNet50 (pretrained)
   Dealer classes: 120
   Model classes: 135
   Total params: 26,263,875
   Trainable params: 26,263,875


In [ ]:
# ============================================================================
# CELL: MODEL SELECTION & VLM/ViT SUPPORT
# ============================================================================

from transformers import CLIPProcessor, CLIPModel, AutoProcessor, BlipForConditionalGeneration
import torch.nn.functional as F

def get_model(model_type, num_dealers, num_models, device):
    if model_type == 'cnn':
        return InvoiceExtractorCNN(num_dealers, num_models).to(device)
    elif model_type == 'clip':
        # Vision-Language Model: CLIP (OpenAI)
        model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
        return model, processor
    elif model_type == 'blip':
        # Vision-Language Model: BLIP (Salesforce)
        model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
        processor = AutoProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        return model, processor
    elif model_type == 'vit':
        # Vision Transformer (ViT)
        from torchvision.models import vit_b_16, ViT_B_16_Weights
        model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        # You may need to adapt the head for multi-task output as in CNN above
        return model.to(device)
    else:
        raise ValueError(f"Unknown model type: {model_type}")

# Example: Choose model type here
MODEL_TYPE = 'cnn'  # Options: 'cnn', 'clip', 'blip', 'vit'

if MODEL_TYPE == 'cnn':
    model = get_model('cnn', len(dealer_encoder.classes_), len(model_encoder.classes_), DEVICE)
elif MODEL_TYPE in ['clip', 'blip']:
    model, processor = get_model(MODEL_TYPE, len(dealer_encoder.classes_), len(model_encoder.classes_), DEVICE)
elif MODEL_TYPE == 'vit':
    model = get_model('vit', len(dealer_encoder.classes_), len(model_encoder.classes_), DEVICE)

print(f"✅ Model loaded: {MODEL_TYPE}")

✅ Model loaded: cnn


In [ ]:
# ============================================================================
# CELL 6: CREATE DATA LOADERS (SUPERVISED ONLY, NO UNLABELED DATA)
# ============================================================================

# Set batch size to 8 for safer GPU memory usage (adjust if OOM)
BATCH_SIZE = 8
train_dataset = InvoiceDataset(train_data, dealer_encoder, model_encoder, is_train=True)
test_dataset = InvoiceDataset(test_data, dealer_encoder, model_encoder, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Print all training image filenames
print(f"\n🖼️ Training images ({len(train_dataset)}):")
for i, item in enumerate(train_dataset):
    print(f"  {i+1:3d}: {item['filename']}")
    if i >= 20 and len(train_dataset) > 25:
        print(f"  ... ({len(train_dataset)-21} more)")
        break

print(f"\n✅ DataLoaders created")
print(f"   Training batches per epoch: {len(train_loader)}")
print(f"   Test batches per epoch: {len(test_loader)}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Training images: {len(train_dataset)}")
print(f"   Test images: {len(test_dataset)}")

# Test one batch
batch = next(iter(train_loader))
print(f"\n📦 Sample batch:")
print(f"   Image shape: {batch['image'].shape}")
print(f"   Dealer indices: {batch['dealer_idx'][:3]}")
print(f"   HP values: {batch['horse_power'][:3]}")


🖼️ Training images (141):
    1: 90019288399_OTHERS_v1_pg1.png
    2: 90019112228_OTHERS_v1_pg1.png
    3: 172571502_1_pg21.png
    4: 90018777756_OTHERS_v1_pg1.png
    5: 90019389310_2_v1_pg1.png
    6: 90019327661_OTHERS_v2_pg1.png
    7: 90018747361_OTHERS_v2_pg1.png
    8: 172926866_3_pg2.png
    9: 173473985_1_pg21.png
   10: 172922192_2_pg35.png
   11: 172934848_2_pg32.png
   12: 90019206687_OTHERS_v1_pg1.png
   13: 173134085_1_pg14.png
   14: 173686375_1_pg40.png
   15: 172864137_2_pg27.png
   16: 173115555_1_pg32.png
   17: 172960101_3_pg1.png
   18: 172656573_1_pg4.png
   19: 90019780900_1_v1_pg1.png
   20: 90019317135_OTHERS_v1_pg1.png
   21: 172937945_3_pg8.png
  ... (120 more)

✅ DataLoaders created
   Training batches per epoch: 18
   Test batches per epoch: 10
   Batch size: 8
   Training images: 141
   Test images: 77

📦 Sample batch:
   Image shape: torch.Size([8, 3, 384, 384])
   Dealer indices: tensor([ 50,  97, 103])
   HP values: tensor([0.4400, 0.4000, 0.0000])


In [ ]:
# Overfit on a small batch
small_batch = next(iter(train_loader))
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()
bce_loss = nn.BCELoss()
for epoch in range(30):
    images = small_batch['image'].to(DEVICE)
    outputs = model(images)
    loss_dealer = ce_loss(outputs['dealer'], small_batch['dealer_idx'].to(DEVICE))
    loss_model = ce_loss(outputs['model'], small_batch['model_idx'].to(DEVICE))
    loss_hp = mse_loss(outputs['horse_power'], small_batch['horse_power'].to(DEVICE))
    loss_cost = mse_loss(outputs['asset_cost'], small_batch['asset_cost'].to(DEVICE))
    loss_sig = bce_loss(outputs['signature'], small_batch['signature'].to(DEVICE))
    loss_stamp = bce_loss(outputs['stamp'], small_batch['stamp'].to(DEVICE))
    loss = (2.0 * loss_dealer + 2.0 * loss_model + 1.0 * loss_hp + 1.0 * loss_cost + 0.5 * loss_sig + 0.5 * loss_stamp)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'Epoch {epoch+1}: Total Loss={loss.item():.4f} | Dealer={loss_dealer.item():.4f} | Model={loss_model.item():.4f} | HP={loss_hp.item():.4f} | Cost={loss_cost.item():.4f} | Sig={loss_sig.item():.4f} | Stamp={loss_stamp.item():.4f}')
    # Check accuracy
    with torch.no_grad():
        dealer_pred = outputs['dealer'].argmax(dim=1)
        model_pred = outputs['model'].argmax(dim=1)
        hp_pred = (outputs['horse_power'] * 100).round()
        cost_pred = (outputs['asset_cost'] * 1000000).round()
        sig_pred = (outputs['signature'] > 0.5).float()
        stamp_pred = (outputs['stamp'] > 0.5).float()
        dealer_gt = small_batch['dealer_idx'].to(DEVICE)
        model_gt = small_batch['model_idx'].to(DEVICE)
        hp_gt = (small_batch['horse_power'] * 100).to(DEVICE)
        cost_gt = (small_batch['asset_cost'] * 1000000).to(DEVICE)
        sig_gt = small_batch['signature'].to(DEVICE)
        stamp_gt = small_batch['stamp'].to(DEVICE)
        acc = ((dealer_pred == dealer_gt).float().mean().item(),
               (model_pred == model_gt).float().mean().item(),
               (torch.abs(hp_pred - hp_gt) <= 5).float().mean().item(),
               (torch.abs(cost_pred - cost_gt) / (cost_gt + 1) <= 0.1).float().mean().item(),
               (sig_pred == sig_gt).float().mean().item(),
               (stamp_pred == stamp_gt).float().mean().item())
        print(f'  Accuracies: Dealer={acc[0]*100:.1f} | Model={acc[1]*100:.1f} | HP={acc[2]*100:.1f} | Cost={acc[3]*100:.1f} | Sig={acc[4]*100:.1f} | Stamp={acc[5]*100:.1f}')
    if loss.item() < 0.1 and all(a > 0.99 for a in acc):
        print('✅ Model can overfit a small batch. Data pipeline and model are OK.')
        break

## 🧪 Debug: Overfit on a Small Batch
Try to overfit the model on 8 samples to check if it can reach near 100% accuracy.

In [ ]:
# Visualize a batch of images and labels
import matplotlib.pyplot as plt
batch = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    if i >= batch['image'].shape[0]: break
    img = batch['image'][i].permute(1,2,0).cpu().numpy()
    img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    img = img.clip(0,1)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"Dealer: {batch['dealer_idx'][i].item()}\nModel: {batch['model_idx'][i].item()}")
plt.tight_layout()
plt.show()
print('Horse Power:', batch['horse_power'][:4])
print('Asset Cost:', batch['asset_cost'][:4])
print('Signature:', batch['signature'][:4])
print('Stamp:', batch['stamp'][:4])

## 🔍 Debug: Visualize a Training Batch and Labels
Check if images and labels are correct before training.

In [ ]:
# ============================================================================
# CELL 7: 🔥 TRAINING LOOP - THE ACTUAL TRAINING!
# ============================================================================

def train_model(model, train_loader, test_loader, epochs=30, lr=1e-4):
    """
    Train the multi-task model.
    This is where the GPU actually does TRAINING!
    """
    
    # Loss functions
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    bce_loss = nn.BCELoss()
    
    # Optimizer with weight decay
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    # Training history
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0
    
    print(f"\n{'='*60}")
    print(f"🔥 STARTING TRAINING - {epochs} epochs")
    print(f"{'='*60}")
    print(f"   Learning rate: {lr}")
    print(f"   Optimizer: AdamW")
    print(f"   Device: {DEVICE}")
    
    for epoch in range(epochs):
        # ===================== TRAINING =====================
        model.train()
        train_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar:
            images = batch['image'].to(DEVICE)
            
            # Forward pass
            outputs = model(images)
            
            # Calculate losses for each task
            loss_dealer = ce_loss(outputs['dealer'], batch['dealer_idx'].to(DEVICE))
            loss_model = ce_loss(outputs['model'], batch['model_idx'].to(DEVICE))
            loss_hp = mse_loss(outputs['horse_power'], batch['horse_power'].to(DEVICE))
            loss_cost = mse_loss(outputs['asset_cost'], batch['asset_cost'].to(DEVICE))
            loss_sig = bce_loss(outputs['signature'], batch['signature'].to(DEVICE))
            loss_stamp = bce_loss(outputs['stamp'], batch['stamp'].to(DEVICE))
            
            # Combined loss (weighted)
            loss = (2.0 * loss_dealer + 2.0 * loss_model + 
                   1.0 * loss_hp + 1.0 * loss_cost +
                   0.5 * loss_sig + 0.5 * loss_stamp)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        train_loss /= len(train_loader)
        scheduler.step()
        
        # ===================== VALIDATION =====================
        model.eval()
        val_loss = 0
        correct = {'dealer': 0, 'model': 0, 'hp': 0, 'cost': 0, 'sig': 0, 'stamp': 0}
        total = 0
        
        with torch.no_grad():
            for batch in test_loader:
                images = batch['image'].to(DEVICE)
                outputs = model(images)
                
                # Predictions
                dealer_pred = outputs['dealer'].argmax(dim=1)
                model_pred = outputs['model'].argmax(dim=1)
                hp_pred = (outputs['horse_power'] * 100).round()
                cost_pred = (outputs['asset_cost'] * 1000000).round()
                sig_pred = (outputs['signature'] > 0.5).float()
                stamp_pred = (outputs['stamp'] > 0.5).float()
                
                # Ground truth
                dealer_gt = batch['dealer_idx'].to(DEVICE)
                model_gt = batch['model_idx'].to(DEVICE)
                hp_gt = (batch['horse_power'] * 100).to(DEVICE)
                cost_gt = (batch['asset_cost'] * 1000000).to(DEVICE)
                sig_gt = batch['signature'].to(DEVICE)
                stamp_gt = batch['stamp'].to(DEVICE)
                
                # Accuracy
                correct['dealer'] += (dealer_pred == dealer_gt).sum().item()
                correct['model'] += (model_pred == model_gt).sum().item()
                correct['hp'] += (torch.abs(hp_pred - hp_gt) <= 5).sum().item()
                correct['cost'] += (torch.abs(cost_pred - cost_gt) / (cost_gt + 1) <= 0.1).sum().item()
                correct['sig'] += (sig_pred == sig_gt).sum().item()
                correct['stamp'] += (stamp_pred == stamp_gt).sum().item()
                total += images.size(0)
        
        # Calculate accuracies
        acc = {k: v / total * 100 for k, v in correct.items()}
        overall_acc = sum(acc.values()) / len(acc)
        
        history['train_loss'].append(train_loss)
        history['val_acc'].append(overall_acc)
        
        print(f"\nEpoch {epoch+1}/{epochs}:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Accuracy: {overall_acc:.1f}%")
        print(f"    Dealer: {acc['dealer']:.1f}% | Model: {acc['model']:.1f}%")
        print(f"    HP: {acc['hp']:.1f}% | Cost: {acc['cost']:.1f}%")
        print(f"    Signature: {acc['sig']:.1f}% | Stamp: {acc['stamp']:.1f}%")
        
        # Save best model
        if overall_acc > best_acc:
            best_acc = overall_acc
            torch.save(model.state_dict(), MODEL_DIR / 'best_model.pt')
            print(f"  ✅ New best model saved! ({best_acc:.1f}%)")
    
    return history, best_acc

# 🔥🔥🔥 START TRAINING 🔥🔥🔥
print("🚀 Training will use T4 GPU for ACTUAL LEARNING!")
history, best_acc = train_model(model, train_loader, test_loader, epochs=30, lr=1e-4)

print(f"\n{'='*60}")
print(f"🏆 TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"Best validation accuracy: {best_acc:.1f}%")

🚀 Training will use T4 GPU for ACTUAL LEARNING!

🔥 STARTING TRAINING - 30 epochs
   Learning rate: 0.0001
   Optimizer: AdamW
   Device: cuda


Epoch 1/30 [Train]:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 1/30 [Train]: 100%|██████████| 18/18 [00:11<00:00,  1.55it/s, loss=20.4310] 



Epoch 1/30:
  Train Loss: 29.8209
  Val Accuracy: 33.1%
    Dealer: 2.6% | Model: 0.0%
    HP: 29.9% | Cost: 23.4%
    Signature: 68.8% | Stamp: 74.0%
  ✅ New best model saved! (33.1%)


Epoch 2/30 [Train]: 100%|██████████| 18/18 [00:11<00:00,  1.61it/s, loss=20.2746]



Epoch 2/30:
  Train Loss: 29.2866
  Val Accuracy: 28.8%
    Dealer: 2.6% | Model: 0.0%
    HP: 20.8% | Cost: 5.2%
    Signature: 68.8% | Stamp: 75.3%


... (52 outputs left out)

Epoch 29/30 [Train]: 100%|██████████| 18/18 [00:10<00:00,  1.73it/s, loss=18.6166]



Epoch 29/30:
  Train Loss: 23.7953
  Val Accuracy: 44.4%
    Dealer: 2.6% | Model: 81.8%
    HP: 16.9% | Cost: 20.8%
    Signature: 68.8% | Stamp: 75.3%


Epoch 30/30 [Train]: 100%|██████████| 18/18 [00:11<00:00,  1.61it/s, loss=14.4189]



Epoch 30/30:
  Train Loss: 23.4172
  Val Accuracy: 45.7%
    Dealer: 2.6% | Model: 84.4%
    HP: 19.5% | Cost: 23.4%
    Signature: 68.8% | Stamp: 75.3%

🏆 TRAINING COMPLETE!
Best validation accuracy: 51.1%


In [ ]:
# ============================================================================
# CELL: TRAINING/FINETUNING FOR VLM/ViT/CNN (FULL EXAMPLE FOR CLIP)
# ============================================================================

if MODEL_TYPE == 'cnn':
    # Use the existing train_model function for CNN
    print("Training CNN model...")
    history, best_acc = train_model(model, train_loader, test_loader, epochs=30, lr=1e-4)

elif MODEL_TYPE == 'vit':
    # For ViT, you need to adapt the head for multi-task output as in CNN, or use transfer learning for classification/regression tasks.
    print("ViT support: Please adapt the model head for multi-task output as in CNN.")

elif MODEL_TYPE == 'clip':
    # CLIP feature extraction + simple MLP head for multi-task output
    from torch.utils.data import Dataset, DataLoader
    class CLIPFeatureDataset(Dataset):
        def __init__(self, data, processor):
            self.data = data
            self.processor = processor
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            img_path, fields = self.data[idx]
            image = Image.open(img_path).convert('RGB')
            inputs = self.processor(images=image, return_tensors="pt")
            # Encode labels as in InvoiceDataset
            dealer_name = fields.get('dealer_name', 'Unknown')
            model_name = fields.get('model_name', 'Unknown')
            dealer_idx = dealer_encoder.transform([dealer_name])[0] if dealer_name in dealer_encoder.classes_ else 0
            model_idx = model_encoder.transform([model_name])[0] if model_name in model_encoder.classes_ else 0
            hp = float(fields.get('horse_power') or 0)
            cost = float(fields.get('asset_cost') or 0)
            sig = 1.0 if fields.get('signature', {}).get('present', False) else 0.0
            stamp = 1.0 if fields.get('stamp', {}).get('present', False) else 0.0
            return {
                'pixel_values': inputs['pixel_values'].squeeze(0),
                'dealer_idx': torch.tensor(dealer_idx, dtype=torch.long),
                'model_idx': torch.tensor(model_idx, dtype=torch.long),
                'horse_power': torch.tensor(hp / 100.0, dtype=torch.float),
                'asset_cost': torch.tensor(cost / 1000000.0, dtype=torch.float),
                'signature': torch.tensor(sig, dtype=torch.float),
                'stamp': torch.tensor(stamp, dtype=torch.float),
            }

    # Prepare CLIP datasets
    train_clip_dataset = CLIPFeatureDataset(train_data, processor)
    test_clip_dataset = CLIPFeatureDataset(test_data, processor)
    train_clip_loader = DataLoader(train_clip_dataset, batch_size=16, shuffle=True, num_workers=2)
    test_clip_loader = DataLoader(test_clip_dataset, batch_size=16, shuffle=False, num_workers=2)

    # Freeze CLIP backbone, add MLP head for multi-task output
    class CLIPMultiTaskHead(nn.Module):
        def __init__(self, clip_embed_dim, num_dealers, num_models):
            super().__init__()
            self.shared = nn.Sequential(
                nn.Linear(clip_embed_dim, 512),
                nn.ReLU(),
                nn.Dropout(0.3),
            )
            self.dealer_head = nn.Linear(512, num_dealers)
            self.model_head = nn.Linear(512, num_models)
            self.hp_head = nn.Linear(512, 1)
            self.cost_head = nn.Linear(512, 1)
            self.signature_head = nn.Linear(512, 1)
            self.stamp_head = nn.Linear(512, 1)
        def forward(self, x):
            shared = self.shared(x)
            return {
                'dealer': self.dealer_head(shared),
                'model': self.model_head(shared),
                'horse_power': self.hp_head(shared).squeeze(-1),
                'asset_cost': self.cost_head(shared).squeeze(-1),
                'signature': torch.sigmoid(self.signature_head(shared)).squeeze(-1),
                'stamp': torch.sigmoid(self.stamp_head(shared)).squeeze(-1),
            }

    # Instantiate head
    clip_embed_dim = model.visual.projection.shape[1] if hasattr(model.visual, 'projection') else 512
    head = CLIPMultiTaskHead(clip_embed_dim, len(dealer_encoder.classes_), len(model_encoder.classes_)).to(DEVICE)

    # Training loop for CLIP+MLP
    optimizer = torch.optim.AdamW(head.parameters(), lr=1e-4, weight_decay=0.01)
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    bce_loss = nn.BCELoss()
    epochs = 10
    for epoch in range(epochs):
        head.train()
        total_loss = 0
        for batch in tqdm(train_clip_loader, desc=f"CLIP+MLP Train Epoch {epoch+1}"):
            with torch.no_grad():
                features = model.get_image_features(batch['pixel_values'].to(DEVICE))
            outputs = head(features)
            loss_dealer = ce_loss(outputs['dealer'], batch['dealer_idx'].to(DEVICE))
            loss_model = ce_loss(outputs['model'], batch['model_idx'].to(DEVICE))
            loss_hp = mse_loss(outputs['horse_power'], batch['horse_power'].to(DEVICE))
            loss_cost = mse_loss(outputs['asset_cost'], batch['asset_cost'].to(DEVICE))
            loss_sig = bce_loss(outputs['signature'], batch['signature'].to(DEVICE))
            loss_stamp = bce_loss(outputs['stamp'], batch['stamp'].to(DEVICE))
            loss = (2.0 * loss_dealer + 2.0 * loss_model + 1.0 * loss_hp + 1.0 * loss_cost + 0.5 * loss_sig + 0.5 * loss_stamp)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_clip_loader):.4f}")

    print("CLIP+MLP training complete. You can now evaluate on the test set using the same head.")

elif MODEL_TYPE == 'blip':
    print("BLIP support: You can use BLIP for image captioning or prompt-based extraction. For field extraction, use BLIP features + MLP head as above.")

Training CNN model...

🔥 STARTING TRAINING - 30 epochs
   Learning rate: 0.0001
   Optimizer: AdamW
   Device: cuda


Epoch 1/30 [Train]: 100%|██████████| 18/18 [00:10<00:00,  1.71it/s, loss=17.5517]



Epoch 1/30:
  Train Loss: 23.4960
  Val Accuracy: 40.3%
    Dealer: 1.3% | Model: 61.0%
    HP: 14.3% | Cost: 20.8%
    Signature: 68.8% | Stamp: 75.3%
  ✅ New best model saved! (40.3%)


Epoch 2/30 [Train]: 100%|██████████| 18/18 [00:11<00:00,  1.56it/s, loss=17.2696]



Epoch 2/30:
  Train Loss: 23.8300
  Val Accuracy: 42.0%
    Dealer: 2.6% | Model: 64.9%
    HP: 16.9% | Cost: 23.4%
    Signature: 68.8% | Stamp: 75.3%
  ✅ New best model saved! (42.0%)


Epoch 3/30 [Train]: 100%|██████████| 18/18 [00:10<00:00,  1.79it/s, loss=18.8029]


... (51 outputs left out)

Epoch 29/30 [Train]: 100%|██████████| 18/18 [00:10<00:00,  1.70it/s, loss=10.7847]



Epoch 29/30:
  Train Loss: 12.8539
  Val Accuracy: 35.7%
    Dealer: 6.5% | Model: 14.3%
    HP: 15.6% | Cost: 28.6%
    Signature: 70.1% | Stamp: 79.2%


Epoch 30/30 [Train]: 100%|██████████| 18/18 [00:11<00:00,  1.63it/s, loss=10.4544]



Epoch 30/30:
  Train Loss: 12.7623
  Val Accuracy: 35.3%
    Dealer: 6.5% | Model: 13.0%
    HP: 18.2% | Cost: 27.3%
    Signature: 68.8% | Stamp: 77.9%


In [ ]:
# ============================================================================
# FINAL EVALUATION ON TEST SET
# ============================================================================

if MODEL_TYPE == 'cnn':
    model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pt'))
    model.eval()
    eval_loader = test_loader
    eval_head = model
elif MODEL_TYPE == 'clip':
    eval_loader = test_clip_loader
    eval_head = head
else:
    raise NotImplementedError("Evaluation for this model type is not implemented.")

print(f"\n📊 FINAL EVALUATION ON TEST SET ({len(test_data)} images)")
print("="*60)

field_correct = defaultdict(int)
total = 0

with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        if MODEL_TYPE == 'cnn':
            images = batch['image'].to(DEVICE)
            outputs = eval_head(images)
        elif MODEL_TYPE == 'clip':
            features = model.get_image_features(batch['pixel_values'].to(DEVICE))
            outputs = eval_head(features)
        
        dealer_pred = outputs['dealer'].argmax(dim=1).cpu().numpy()
        model_pred = outputs['model'].argmax(dim=1).cpu().numpy()
        hp_pred = (outputs['horse_power'].cpu().numpy() * 100).round()
        cost_pred = (outputs['asset_cost'].cpu().numpy() * 1000000).round()
        sig_pred = (outputs['signature'].cpu().numpy() > 0.5)
        stamp_pred = (outputs['stamp'].cpu().numpy() > 0.5)
        
        dealer_gt = batch['dealer_idx'].numpy()
        model_gt = batch['model_idx'].numpy()
        hp_gt = (batch['horse_power'].numpy() * 100).round()
        cost_gt = (batch['asset_cost'].numpy() * 1000000).round()
        sig_gt = batch['signature'].numpy() > 0.5
        stamp_gt = batch['stamp'].numpy() > 0.5
        
        for i in range(len(dealer_pred)):
            dealer_match = dealer_pred[i] == dealer_gt[i]
            model_match = model_pred[i] == model_gt[i]
            hp_match = abs(hp_pred[i] - hp_gt[i]) <= 5
            cost_match = abs(cost_pred[i] - cost_gt[i]) / max(cost_gt[i], 1) <= 0.10
            sig_match = sig_pred[i] == sig_gt[i]
            stamp_match = stamp_pred[i] == stamp_gt[i]
            field_correct['dealer_name'] += dealer_match
            field_correct['model_name'] += model_match
            field_correct['horse_power'] += hp_match
            field_correct['asset_cost'] += cost_match
            field_correct['signature'] += sig_match
            field_correct['stamp'] += stamp_match
            total += 1

field_accuracy = {k: v / total * 100 for k, v in field_correct.items()}
overall_accuracy = sum(field_accuracy.values()) / len(field_accuracy)

print(f"\n{'='*60}")
print(f"📈 OVERALL ACCURACY: {overall_accuracy:.1f}%")
print(f"{'='*60}")
print(f"\n📋 Per-Field Accuracy:")
for field, acc in field_accuracy.items():
    status = "✅" if acc >= 90 else "⚠️" if acc >= 70 else "❌"
    print(f"   {status} {field:20s}: {acc:5.1f}%")


📊 FINAL EVALUATION ON TEST SET (77 images)


Evaluating: 100%|██████████| 10/10 [00:08<00:00,  1.20it/s]


📈 OVERALL ACCURACY: 43.3%

📋 Per-Field Accuracy:
   ❌ dealer_name         :   1.3%
   ❌ model_name          :  67.5%
   ❌ horse_power         :  18.2%
   ❌ asset_cost          :  28.6%
   ❌ signature           :  68.8%
   ⚠️ stamp               :  75.3%


In [ ]:
# ============================================================================
# VISUALIZE PREDICTIONS AND ACCURACY PER IMAGE
# ============================================================================
from IPython.display import display, Image as IPImage, JSON

def visualize_predictions(eval_head, eval_loader, dealer_encoder, model_encoder, device=DEVICE):
    eval_head.eval()
    total = 0
    overall_correct = 0
    with torch.no_grad():
        for batch in eval_loader:
            if MODEL_TYPE == 'cnn':
                images = batch['image'].to(device)
                outputs = eval_head(images)
                filenames = batch['filename']
            elif MODEL_TYPE == 'clip':
                features = model.get_image_features(batch['pixel_values'].to(device))
                outputs = eval_head(features)
                filenames = [None]*len(features)  # CLIPFeatureDataset does not provide filenames by default
            else:
                continue
            dealer_pred = outputs['dealer'].argmax(dim=1).cpu().numpy()
            model_pred = outputs['model'].argmax(dim=1).cpu().numpy()
            hp_pred = (outputs['horse_power'].cpu().numpy() * 100).round()
            cost_pred = (outputs['asset_cost'].cpu().numpy() * 1000000).round()
            sig_pred = (outputs['signature'].cpu().numpy() > 0.5)
            stamp_pred = (outputs['stamp'].cpu().numpy() > 0.5)
            dealer_gt = batch['dealer_idx'].numpy()
            model_gt = batch['model_idx'].numpy()
            hp_gt = (batch['horse_power'].numpy() * 100).round()
            cost_gt = (batch['asset_cost'].numpy() * 1000000).round()
            sig_gt = batch['signature'].numpy() > 0.5
            stamp_gt = batch['stamp'].numpy() > 0.5
            for i in range(len(dealer_pred)):
                pred_json = {
                    'dealer_name': dealer_encoder.classes_[dealer_pred[i]],
                    'model_name': model_encoder.classes_[model_pred[i]],
                    'horse_power': int(hp_pred[i]),
                    'asset_cost': int(cost_pred[i]),
                    'signature': {'present': bool(sig_pred[i])},
                    'stamp': {'present': bool(stamp_pred[i])},
                }
                gt_json = {
                    'dealer_name': dealer_encoder.classes_[dealer_gt[i]],
                    'model_name': model_encoder.classes_[model_gt[i]],
                    'horse_power': int(hp_gt[i]),
                    'asset_cost': int(cost_gt[i]),
                    'signature': {'present': bool(sig_gt[i])},
                    'stamp': {'present': bool(stamp_gt[i])},
                }
                accs = {}
                accs['dealer_name'] = int(pred_json['dealer_name'] == gt_json['dealer_name'])
                accs['model_name'] = int(pred_json['model_name'] == gt_json['model_name'])
                accs['horse_power'] = int(abs(pred_json['horse_power'] - gt_json['horse_power']) <= 5)
                accs['asset_cost'] = int(abs(pred_json['asset_cost'] - gt_json['asset_cost']) / max(gt_json['asset_cost'], 1) <= 0.10)
                accs['signature'] = int(pred_json['signature']['present'] == gt_json['signature']['present'])
                accs['stamp'] = int(pred_json['stamp']['present'] == gt_json['stamp']['present'])
                per_image_acc = sum(accs.values()) / len(accs) * 100
                overall_correct += (per_image_acc == 100)
                total += 1
                print(f"\n{'='*60}")
                print(f"Image: {filenames[i] if filenames[i] else 'N/A'}")
                if filenames[i]: display(IPImage(str(TRAIN_DIR / filenames[i]), width=400))
                print(f"Predicted JSON:")
                display(JSON(pred_json))
                print(f"Ground Truth JSON:")
                display(JSON(gt_json))
                print(f"Per-field accuracy: {accs}")
                print(f"Per-image accuracy: {per_image_acc:.1f}%")
                print(f"{'='*60}")
    print(f"\n{'#'*60}")
    print(f"OVERALL: {overall_correct}/{total} images were 100% correct")
    print(f"Overall perfect accuracy: {overall_correct/total*100:.1f}%")